In [24]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd
from tqdm import tqdm

In [25]:
# Configuracion
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

Usando dispositivo: cpu


In [26]:
# 1. DATOS DE EJEMPLO (comentarios en español)
comentarios = [
    # Positivos (1)
    "Me encantó este producto, es excelente",
    "Muy buen servicio, lo recomiendo totalmente",
    "La atención al cliente fue increíble",
    "Excelente calidad, superó mis expectativas",
    "Me siento muy satisfecho con mi compra",
    "El producto llegó antes de lo esperado",
    "Totalmente recomendado, funciona perfecto",
    "La mejor compra que he hecho este año",
    "El personal es muy amable y profesional",
    "Volvería a comprar sin dudarlo",
    
    # Negativos (0)
    "Pésimo servicio, no lo recomiendo",
    "El producto llegó roto y en mal estado",
    "Muy mala experiencia, no volveré a comprar",
    "La atención al cliente es terrible",
    "No funciona como esperaba, decepcionado",
    "El envío tardó mucho más de lo debido",
    "Producto de baja calidad, no vale lo que cuesta",
    "Me arrepiento de haber comprado esto",
    "El servicio al cliente no responde",
    "No cumple con lo prometido, muy malo"
]


In [27]:
etiquetas = [1] * 10 + [0] * 10  # 1 para positivos, 0 para negativos
etiquetas

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [28]:
# Crea dataFrame
df = pd.DataFrame({'comentarios': comentarios, 'sentimiento': etiquetas})
df

,comentarios,sentimiento
0,"Me encantó este producto, es excelente",1
1,"Muy buen servicio, lo recomiendo totalmente",1
2,La atención al cliente fue increíble,1
3,"Excelente calidad, superó mis expectativas",1
4,Me siento muy satisfecho con mi compra,1
5,El producto llegó antes de lo esperado,1
6,"Totalmente recomendado, funciona perfecto",1
7,La mejor compra que he hecho este año,1
8,El personal es muy amable y profesional,1
9,Volvería a comprar sin dudarlo,1


## DataSet personalizado para PyTorch

In [29]:
class SentimentDataset(Dataset):
    def __init__(self, comentarios, etiquetas, tokenizer, max_len=120):
        self.comentarios = comentarios
        self.etiquetas = etiquetas
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.comentarios)

    def __getitem__(self, idx):
        texto = str(self.comentarios[idx])
        etiqueta = self.etiquetas[idx]

        # Tokenizar el texto
        encoding = self.tokenizer(
            texto,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(int(etiqueta), dtype=torch.long)
        }

In [30]:
# 3. Preparar datos
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [31]:
# Dividir datos en entrenamiento y puruebas
x_train, x_test, y_train, y_test = train_test_split(
    df['comentarios'].values,
    df['sentimiento'].values,
    test_size=0.3,
    random_state=23,
    stratify=df['sentimiento'].values
)

In [32]:
print(f'Tamaño del conjunto de entrenamiento: {len(x_train)}')
print(f'Tamaño del conjunto de prueba: {len(x_test)}')

Tamaño del conjunto de entrenamiento: 14
Tamaño del conjunto de prueba: 6


In [33]:
train_dataset = SentimentDataset(x_train, y_train, tokenizer)
test_dataset = SentimentDataset(x_test, y_test, tokenizer)
test_dataset


In [34]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=True)

## Entrenamiento del modelo

In [ ]:
modelo = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
modelo = modelo.to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\chriv\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chriv\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

You are using a model of type `distilbert` to instantiate a model of type `bert`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                                                                      | Status     | 
-------------------------------------------------------------------------+------------+-
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.sa_layer_norm.weight     | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.weight   | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.out_lin.weight | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.q_lin.bias     | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.bias     | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.k_lin.weight   | UNEXPECTED | 
distilbert.embeddings.word_embeddings.weight                             | UNEXPECTED | 
distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.ffn.lin2.weight          | UNEXPECTED | 
distilbert.tra

In [36]:
optimizador = AdamW(modelo.parameters(), lr=2e-5)

In [37]:
# Funcion de entrenamiento
def entrenar_modelo(modelo, train_dataloader, test_dataloader, optimizer, epocas=5):
    train_losses = []
    test_accuracies = []
    for epoca in range(epocas):
        print(f'Epoca {epoca + 1}/{epocas}: ')
        print('=' * 50)
        # Modo entrenamiento
        modelo.train()
        total_loss = 0
        
        for batch in tqdm(train_dataloader, desc='Entrenando'):
            # Mover datos al dispositivo
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            ## Pasar
            optimizer.zero_grad()
            outputs = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss
            
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(train_dataloader)
        train_losses.append(avg_loss)
            
        # Evaluacion
        accuracy = evaluar_modelo(modelo, test_dataloader)
        test_accuracies.append(accuracy)
        print(f'Promedio de perdidas: {avg_loss:.4f} | Accuracy en prueba: {accuracy:.4f}')
            
    return train_losses, test_accuracies

In [38]:
def evaluar_modelo(modelo, test_dataloader):
    modelo.eval()
    predicciones = []
    verdaderos = []
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc='Evaluando'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = modelo(input_ids=input_ids, attention_mask=attention_mask)
            _, pred = torch.max(outputs.logits, dim=1)
            predicciones.extend(pred.cpu().numpy())
            verdaderos.extend(labels.cpu().numpy())
            
    accuracy = accuracy_score(verdaderos, predicciones)
    return accuracy
            

In [39]:
# Entranar el modelo
train_losses, test_accuracies = entrenar_modelo(
    modelo, train_dataloader, test_dataloader, optimizador, epocas=5
)

Epoca 1/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.26it/s]


Promedio de perdidas: 1.1274 | Accuracy en prueba: 0.5000
Epoca 2/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.40it/s]


Promedio de perdidas: 0.9263 | Accuracy en prueba: 0.5000
Epoca 3/5: 


...

Promedio de perdidas: 0.8458 | Accuracy en prueba: 0.5000
Epoca 4/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.38it/s]


Promedio de perdidas: 0.6809 | Accuracy en prueba: 0.5000
Epoca 5/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.33it/s]

Promedio de perdidas: 0.6899 | Accuracy en prueba: 0.5000


In [40]:
# Evaluación final detallada
modelo.eval()
all_predicciones = []
all_verdaderos = []
all_probabilidades = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = modelo(
        input_ids=input_ids,
        attention_mask=attention_mask
        )

        # Obtener probabilidades con softmax
        probabilidades = torch.nn.functional.softmax(outputs.logits, dim=1)
        _, preds = torch.max(outputs.logits, dim=1)

        all_predicciones.extend(preds.cpu().tolist())
        all_verdaderos.extend(labels.tolist())
        all_probabilidades.extend(probabilidades.cpu().tolist())

        # Reporte de clasificación
        print("\n" + "="*50)
        print("REPORTE DE CLASIFICACIÓN")
        print("="*50)
        print(classification_report(
        all_verdaderos,
        all_predicciones,
        target_names=['Negativo', 'Positivo']
        ))

# Mostrar algunos ejemplos con predicciones
print("\n" + "="*50)
print("EJEMPLOS DE PREDICCIONES")
print("="*50)
indices_prueba = np.random.choice(len(x_test), size=5, replace=False)
for idx in indices_prueba:
    comentario = x_test[idx]
    real = "POSITIVO" if y_test[idx] == 1 else "NEGATIVO"
    pred = "POSITIVO" if all_predicciones[idx] == 1 else "NEGATIVO"
    prob_pos = all_probabilidades[idx][1]
    prob_neg = all_probabilidades[idx][0]

    print(f"\nComentario: {comentario}")
    print(f"Real: {real}")
    print(f"Predicción: {pred}")
    print(f"Probabilidad Positivo: {prob_pos:.4f}")
    print(f"Probabilidad Negativo: {prob_neg:.4f}")
    print(f"{'✓' if real == pred else '✗'} Correcto: {real == pred}")


REPORTE DE CLASIFICACIÓN
              precision    recall  f1-score   support

    Negativo       0.50      1.00      0.67         2
    Positivo       0.00      0.00      0.00         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4



C:\Users\chriv\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\chriv\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\chriv\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital


REPORTE DE CLASIFICACIÓN
              precision    recall  f1-score   support

    Negativo       0.50      1.00      0.67         3
    Positivo       0.00      0.00      0.00         3

    accuracy                           0.50         6
   macro avg       0.25      0.50      0.33         6
weighted avg       0.25      0.50      0.33         6


EJEMPLOS DE PREDICCIONES

Comentario: Totalmente recomendado, funciona perfecto
Real: POSITIVO
Predicción: NEGATIVO
Probabilidad Positivo: 0.4612
Probabilidad Negativo: 0.5388
✗ Correcto: False

Comentario: La atención al cliente es terrible
Real: NEGATIVO
Predicción: NEGATIVO
Probabilidad Positivo: 0.4772
Probabilidad Negativo: 0.5228
✓ Correcto: True

Comentario: Excelente calidad, superó mis expectativas
Real: POSITIVO
Predicción: NEGATIVO
Probabilidad Positivo: 0.4584
Probabilidad Negativo: 0.5416
✗ Correcto: False

Comentario: Volvería a comprar sin dudarlo
Real: POSITIVO
Predicción: NEGATIVO
Probabilidad Positivo: 0.4627
Probabilida

C:\Users\chriv\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\chriv\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\chriv\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

## Función para nuevos comentarios

In [41]:
def predecir_sentimiento(texto, model, tokenizer, device):
    """
    Predice el sentimiento de un nuevo comentario
    """
    modelo.eval()
    
    # Tokenizar
    encoding = tokenizer(
        texto,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Mover a dispositivo
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predecir
    with torch.no_grad():
        outputs = modelo(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        probabilidades = torch.nn.functional.softmax(outputs.logits, dim=1)
        prediccion = torch.argmax(probabilidades, dim=1)
        
        prob_pos = probabilidades[0][1].item()
        prob_neg = probabilidades[0][0].item()
    
    sentimiento = "POSITIVO" if prediccion.item() == 1 else "NEGATIVO"
    confianza = max(prob_pos, prob_neg)
    
    return {
        'sentimiento': sentimiento,
        'confianza': confianza,
        'probabilidad_positivo': prob_pos,
        'probabilidad_negativo': prob_neg
    }

## Listado de nuevos comentarios

In [42]:
# Probar con nuevos comentarios
print("\n" + "="*50)
print("PREDICCIONES EN TIEMPO REAL")
print("="*50)
nuevos_comentarios = [
    "Este producto es increíble, me encantó",
    "Una verdadera porquería, no funciona",
    "Más o menos, podría ser mejor",
    "Excelente atención, muy recomendable",
    "No me gustó para nada, muy decepcionante"
]

for comentario in nuevos_comentarios:
    resultado = predecir_sentimiento(comentario, modelo, tokenizer, device)
    print(f"\nComentario: {comentario}")
    print(f"Sentimiento: {resultado['sentimiento']}")
    print(f"Confianza: {resultado['confianza']:.2%}")
    print(f"Positivo: {resultado['probabilidad_positivo']:.2%}, "
          f"Negativo: {resultado['probabilidad_negativo']:.2%}")


PREDICCIONES EN TIEMPO REAL

Comentario: Este producto es increíble, me encantó
Sentimiento: NEGATIVO
Confianza: 52.02%
Positivo: 47.98%, Negativo: 52.02%

Comentario: Una verdadera porquería, no funciona
Sentimiento: NEGATIVO
Confianza: 54.43%
Positivo: 45.57%, Negativo: 54.43%

Comentario: Más o menos, podría ser mejor
Sentimiento: NEGATIVO
Confianza: 53.58%
Positivo: 46.42%, Negativo: 53.58%

Comentario: Excelente atención, muy recomendable
Sentimiento: NEGATIVO
Confianza: 52.99%
Positivo: 47.01%, Negativo: 52.99%

Comentario: No me gustó para nada, muy decepcionante
Sentimiento: NEGATIVO
Confianza: 53.67%
Positivo: 46.33%, Negativo: 53.67%


In [43]:
comentario = 'Me encanta, lo volvería a comprar'
resultado = predecir_sentimiento(comentario, modelo, tokenizer, device)
print(f"\nComentario: {comentario}")
print(f"Sentimiento: {resultado['sentimiento']}")
print(f"Confianza: {resultado['confianza']:.2%}")
print(f"Positivo: {resultado['probabilidad_positivo']:.2%}, "
      f"Negativo: {resultado['probabilidad_negativo']:.2%}")


Comentario: Me encanta, lo volvería a comprar
Sentimiento: NEGATIVO
Confianza: 54.03%
Positivo: 45.97%, Negativo: 54.03%


In [ ]:
import tkinter as tk
from tkinter import ttk


def analizar_texto():
    texto_ingresado = entrada_texto.get("1.0", tk.END).strip()
    
    if not texto_ingresado:
        resultado_label.config(text="Por favor, ingresa un comentario.", fg="red")
        return
    resultado = predecir_sentimiento(texto_ingresado, modelo, tokenizer, device)
    
    color = "green" if resultado['sentimiento'] == "POSITIVO" else "red"
    
    texto_resultado = (
        f"Sentimiento: {resultado['sentimiento']}\n"
        f"Confianza: {resultado['confianza']:.2%}\n\n"
        f"Detalle:\n"
        f"Positivo: {resultado['probabilidad_positivo']:.2%}\n"
        f"Negativo: {resultado['probabilidad_negativo']:.2%}"
    )
    
    # Actualizar la etiqueta con los resultados
    resultado_label.config(text=texto_resultado, fg=color)

ventana = tk.Tk()
ventana.title("Analizador de Sentimientos NLP")
ventana.geometry("400x350")
ventana.configure(padx=20, pady=20)

titulo = tk.Label(ventana, text="Análisis de Sentimientos", font=("Arial", 14, "bold"))
titulo.pack(pady=(0, 10))

instruccion = tk.Label(ventana, text="Ingresa un comentario:")
instruccion.pack(anchor="w")

entrada_texto = tk.Text(ventana, height=4, width=40, font=("Arial", 11))
entrada_texto.pack(pady=5)

boton_analizar = tk.Button(ventana, text="Analizar Sentimiento", command=analizar_texto, bg="#0078D7", fg="white", font=("Arial", 11, "bold"))
boton_analizar.pack(pady=10)

resultado_label = tk.Label(ventana, text="", font=("Arial", 11), justify="left")
resultado_label.pack(pady=10)

ventana.mainloop()